# Introdução à Modelagem de Risco de Crédito
## Parte 2 — do modelo à decisão

Este notebook é a continuação prática da Parte 1. Na Parte 1, estruturamos o problema, preparamos os dados, criamos variáveis e discutimos discretização, WoE e IV.

Nesta parte, vamos usar os dados preparados para:

- ajustar uma regressão logística;
- transformar probabilidades em classificações;
- avaliar o modelo com matriz de confusão, ROC, AUC e Precision–Recall;
- discutir o papel do threshold;
- comparar a logística com modelos alternativos.

> Ideia central: o modelo estima risco, mas a decisão depende de como usamos esse risco.

## 1. Preparação do ambiente

Vamos carregar os pacotes necessários. O notebook é autocontido: ele carrega a base novamente para que possa ser executado no Colab sem depender da Parte 1 aberta.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    brier_score_loss
)

RANDOM_STATE = 42

## 2. Carregando a base

Usaremos a base `credit-g`, disponível no OpenML. A variável original `class` tem dois níveis:

- `good`: cliente adimplente;
- `bad`: cliente inadimplente.

Vamos criar a variável `target`:

- `target = 1`: inadimplente;
- `target = 0`: adimplente.

In [ ]:
credit = fetch_openml(name="credit-g", version=1, as_frame=True)

X_original = credit.data.copy()
y_original = credit.target.copy()

dados = X_original.copy()
dados["class"] = y_original

# Variável alvo: 1 = inadimplente, 0 = adimplente
dados["target"] = (dados["class"] == "bad").astype(int)

dados.head()

### Conferência rápida da variável alvo

Antes de modelar, sempre confira a distribuição da resposta. Isso ajuda a lembrar que o problema é desbalanceado: normalmente há mais clientes adimplentes do que inadimplentes.

In [ ]:
contagem = dados["target"].value_counts().sort_index()
proporcao = dados["target"].value_counts(normalize=True).sort_index()

resumo_alvo = pd.DataFrame({
    "classe": ["Adimplente (0)", "Inadimplente (1)"],
    "quantidade": contagem.values,
    "proporcao": proporcao.values
})

resumo_alvo

## 3. Retomando a preparação feita na Parte 1

Na Parte 1, discutimos que modelar não começa no algoritmo. Antes disso, precisamos preparar os dados.

Aqui vamos manter duas transformações simples:

- `log_credit_amount`: logaritmo do valor do crédito;
- `credit_per_month`: valor médio do crédito por mês.

Essas variáveis ajudam a representar melhor o risco associado ao valor e à duração do crédito.

In [ ]:
dados["log_credit_amount"] = np.log1p(dados["credit_amount"])
dados["credit_per_month"] = dados["credit_amount"] / dados["duration"]

dados[["credit_amount", "duration", "log_credit_amount", "credit_per_month"]].head()

### Checagens mínimas antes do modelo

Mesmo em uma base didática, é importante verificar:

- dados faltantes;
- duplicatas;
- tipos das variáveis.

In [ ]:
print("Dados faltantes nas 10 primeiras colunas com mais ausências:")
display(dados.isna().sum().sort_values(ascending=False).head(10))

print("\nNúmero de linhas duplicadas:", dados.duplicated().sum())

print("\nTipos das primeiras variáveis:")
display(dados.dtypes.head(15))

## 4. Divisão dos dados

Vamos repetir a divisão discutida na Parte 1.

A base será dividida em:

- treino: usado para ajustar o modelo;
- validação: útil para comparação e ajuste de decisões;
- teste: usado para avaliação final.

Neste notebook, usaremos principalmente treino e teste para manter o fluxo simples, mas manteremos a validação disponível.

In [ ]:
X = dados.drop(columns=["class", "target"])
y = dados["target"]

# 1ª divisão: 80% treino inicial, 20% teste
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# 2ª divisão: dos 80%, separar 75% treino e 25% validação
# Resultado final aproximado: 60% treino, 20% validação, 20% teste
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("Treino:", X_train.shape[0])
print("Validação:", X_valid.shape[0])
print("Teste:", X_test.shape[0])

### Conferindo a estratificação

Como o problema é desbalanceado, usamos `stratify` para preservar a proporção de inadimplentes nos conjuntos.

In [ ]:
proporcoes = pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "proporcao_inadimplentes": [y_train.mean(), y_valid.mean(), y_test.mean()]
})

proporcoes

## 5. Modelo logístico: ideia prática

A regressão logística estima, para cada cliente, uma probabilidade de inadimplência.

Em termos práticos:

- entrada: características do cliente;
- saída: probabilidade entre 0 e 1;
- interpretação: risco estimado.

> O modelo não decide. Ele estima risco.

## 6. Ajustando a regressão logística

Como a base tem variáveis numéricas e categóricas, precisamos transformar as categóricas em variáveis numéricas.

No notebook, vamos usar um `Pipeline`, que organiza:

1. tratamento das variáveis categóricas;
2. manutenção das variáveis numéricas;
3. ajuste da regressão logística.

Esse fluxo é mais robusto e evita erros entre treino e teste.

In [ ]:
# Identificar colunas categóricas e numéricas
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

modelo_log = Pipeline(steps=[
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])

modelo_log.fit(X_train, y_train)

y_prob_log = modelo_log.predict_proba(X_test)[:, 1]

print("Primeiras probabilidades previstas:")
print(y_prob_log[:10])

### O que o código fez?

- transformou variáveis categóricas em numéricas;
- ajustou o modelo usando os dados de treino;
- aplicou o modelo aos dados de teste;
- retornou uma probabilidade de inadimplência para cada cliente no teste.

> Agora podemos avaliar se essas probabilidades são úteis.

## 7. Visualizando as probabilidades previstas

Vamos observar a distribuição das probabilidades estimadas pelo modelo.

A ideia é verificar se o modelo atribui probabilidades diferentes para clientes adimplentes e inadimplentes.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(y_prob_log[y_test.values == 0], bins=20, alpha=0.65, label="Adimplentes (0)")
plt.hist(y_prob_log[y_test.values == 1], bins=20, alpha=0.65, label="Inadimplentes (1)")
plt.xlabel("Probabilidade prevista de inadimplência")
plt.ylabel("Frequência")
plt.title("Distribuição das probabilidades previstas")
plt.legend()
plt.show()

### Atividade de leitura

Observe o gráfico anterior e responda:

1. As distribuições de adimplentes e inadimplentes estão bem separadas?
2. Há sobreposição entre os grupos?
3. O que essa sobreposição significa para a decisão de crédito?

## 8. Da probabilidade para a classificação

Para construir uma matriz de confusão, precisamos transformar probabilidades em classes.

Usaremos inicialmente o ponto de corte 0,5:

- probabilidade maior ou igual a 0,5 → classifica como inadimplente;
- probabilidade menor que 0,5 → classifica como adimplente.

Esse ponto de corte é apenas uma escolha inicial.

In [ ]:
threshold = 0.5
y_pred_log = (y_prob_log >= threshold).astype(int)

print("Quantidade prevista como adimplente (0):", (y_pred_log == 0).sum())
print("Quantidade prevista como inadimplente (1):", (y_pred_log == 1).sum())

## 9. Matriz de confusão

A matriz de confusão compara:

- o que realmente aconteceu;
- o que o modelo previu.

Ela permite visualizar acertos e erros.

In [ ]:
cm = confusion_matrix(y_test, y_pred_log)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Adimplente (0)", "Inadimplente (1)"]
)

disp.plot(values_format="d")
plt.title("Matriz de confusão - Regressão logística")
plt.show()

### Interpretando a matriz

- Verdadeiro negativo: cliente adimplente corretamente classificado como adimplente.
- Verdadeiro positivo: cliente inadimplente corretamente classificado como inadimplente.
- Falso positivo: cliente adimplente classificado como inadimplente.
- Falso negativo: cliente inadimplente classificado como adimplente.

Em crédito, esses erros têm impactos diferentes.

## 10. Métricas básicas

Vamos calcular algumas métricas a partir da classificação com threshold 0,5.

In [ ]:
print(classification_report(
    y_test,
    y_pred_log,
    target_names=["Adimplente (0)", "Inadimplente (1)"],
    digits=3
))

### Como ler essas métricas?

- acurácia: proporção total de acertos;
- precisão: entre os classificados como inadimplentes, quantos realmente eram inadimplentes;
- recall: entre os inadimplentes reais, quantos foram identificados.

> Em crédito, a classe inadimplente costuma ser a mais importante para a análise de risco.

## 11. Curva ROC e AUC

A curva ROC avalia o desempenho do modelo para diferentes valores de threshold.

Ela relaciona:

- TPR: taxa de verdadeiros positivos;
- FPR: taxa de falsos positivos.

A AUC resume a capacidade de separação do modelo.

In [ ]:
fpr_log, tpr_log, thresholds_log = roc_curve(y_test, y_prob_log)
auc_log = roc_auc_score(y_test, y_prob_log)

plt.figure(figsize=(7, 5))
plt.plot(fpr_log, tpr_log, label=f"Logística (AUC = {auc_log:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Aleatório")
plt.xlabel("FPR - taxa de falsos positivos")
plt.ylabel("TPR - taxa de verdadeiros positivos")
plt.title("Curva ROC")
plt.legend()
plt.show()

print("AUC:", round(auc_log, 3))

### Interpretação da AUC

A AUC mede a capacidade do modelo de ordenar clientes por risco.

Uma interpretação útil:

> se escolhermos um cliente inadimplente e um adimplente ao acaso, a AUC mede a chance de o modelo atribuir maior risco ao inadimplente.

Importante: AUC avalia separação, não garante boa decisão nem boa calibração.

## 12. Precision–Recall

A curva Precision–Recall é útil quando a classe de interesse é rara.

No nosso caso, a classe positiva é:

- `target = 1`: inadimplente.

In [ ]:
precision_log, recall_log, thresholds_pr_log = precision_recall_curve(y_test, y_prob_log)
ap_log = average_precision_score(y_test, y_prob_log)

plt.figure(figsize=(7, 5))
plt.plot(recall_log, precision_log, label=f"Logística (AP = {ap_log:.3f})")
plt.xlabel("Recall - inadimplentes identificados")
plt.ylabel("Precisão - acerto entre os classificados como risco")
plt.title("Curva Precision–Recall")
plt.legend()
plt.show()

print("Average Precision:", round(ap_log, 3))

### Atividade de interpretação

Compare ROC e Precision–Recall:

1. Qual delas parece mais conectada à classe inadimplente?
2. Por que Precision–Recall pode ser útil em dados desbalanceados?
3. Uma boa AUC garante uma boa decisão de crédito?

## 13. Calibração e Brier score

Além de separar bons e maus clientes, queremos saber se as probabilidades fazem sentido.

O Brier score mede o erro médio quadrático das probabilidades previstas:

- menor Brier → probabilidades mais próximas dos valores observados;
- maior Brier → probabilidades piores.

> Um modelo pode ranquear bem, mas ainda produzir probabilidades mal calibradas.

In [ ]:
brier_log = brier_score_loss(y_test, y_prob_log)
print("Brier score - Logística:", round(brier_log, 4))

## 14. Efeito do threshold

O threshold controla quantos clientes serão classificados como risco.

Vamos comparar alguns pontos de corte.

In [ ]:
def resumo_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "classificados_risco": int((y_pred == 1).sum()),
        "recall_inad": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "precisao_inad": tp / (tp + fp) if (tp + fp) > 0 else np.nan
    }

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
resumo = pd.DataFrame([resumo_threshold(y_test, y_prob_log, t) for t in thresholds])
resumo

### Como interpretar?

Ao reduzir o threshold:

- mais clientes são classificados como risco;
- aumenta a chance de identificar inadimplentes;
- também pode aumentar falsos alarmes.

Ao aumentar o threshold:

- menos clientes são classificados como risco;
- a decisão fica mais restritiva;
- alguns inadimplentes podem passar sem identificação.

## 15. Threshold, erro e custo

Na Parte 1, discutimos a ideia:

\[
\text{Custo} = c_{FP} \cdot FP + c_{FN} \cdot FN
\]

Agora vamos transformar essa ideia em cálculo simples.

Suponha:

- custo de falso positivo: 100;
- custo de falso negativo: 800.

Esses valores são apenas ilustrativos.

In [ ]:
def custo_threshold(y_true, y_prob, threshold, custo_fp=100, custo_fn=800):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    custo = custo_fp * fp + custo_fn * fn
    return {
        "threshold": threshold,
        "FP": fp,
        "FN": fn,
        "custo_total": custo
    }

custos = pd.DataFrame([
    custo_threshold(y_test, y_prob_log, t, custo_fp=100, custo_fn=800)
    for t in np.arange(0.1, 0.91, 0.1)
])

custos

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(custos["threshold"], custos["custo_total"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Custo total")
plt.title("Custo total para diferentes thresholds")
plt.show()

melhor = custos.loc[custos["custo_total"].idxmin()]
print("Threshold com menor custo:", melhor["threshold"])
print("Custo mínimo:", melhor["custo_total"])

### Atividade

Altere os custos no código anterior e responda:

1. O melhor threshold mudou?
2. O que acontece quando o custo de deixar passar inadimplentes aumenta?
3. Por que o threshold 0,5 pode não ser adequado em crédito?

## 16. Outros modelos

Agora vamos comparar a regressão logística com modelos alternativos:

- árvore de decisão;
- random forest;
- gradient boosting.

A pergunta não é apenas “qual tem maior métrica?”, mas:

> qual modelo ajuda a tomar uma decisão melhor?

In [ ]:
modelo_tree = Pipeline(steps=[
    ("prep", preprocess),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
])

modelo_rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))
])

modelo_gb = Pipeline(steps=[
    ("prep", preprocess),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
])

modelos = {
    "Logística": modelo_log,
    "Árvore": modelo_tree,
    "Random Forest": modelo_rf,
    "Gradient Boosting": modelo_gb
}

probs = {}

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    probs[nome] = modelo.predict_proba(X_test)[:, 1]

print("Modelos ajustados com sucesso.")

## 17. Comparando modelos por AUC e Brier

Vamos comparar duas dimensões:

- AUC: capacidade de separação;
- Brier score: qualidade das probabilidades.

In [ ]:
comparacao = []

for nome, y_prob in probs.items():
    comparacao.append({
        "modelo": nome,
        "AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob)
    })

comparacao = pd.DataFrame(comparacao).sort_values("AUC", ascending=False)
comparacao

### Atenção

O modelo com maior AUC nem sempre será o melhor para decisão.

A escolha também depende de:

- interpretabilidade;
- estabilidade;
- custo dos erros;
- exigências de governança.

## 18. Curvas ROC comparadas

Agora vamos visualizar a capacidade de separação dos modelos.

In [ ]:
plt.figure(figsize=(7, 5))

for nome, y_prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Aleatório")
plt.xlabel("FPR - taxa de falsos positivos")
plt.ylabel("TPR - taxa de verdadeiros positivos")
plt.title("Comparação de modelos - Curva ROC")
plt.legend()
plt.show()

## 19. Comparação com threshold fixo

Vamos comparar os modelos usando o mesmo threshold inicial: 0,5.

Isso mostra como a escolha do modelo afeta a matriz de confusão.

In [ ]:
linhas = []

for nome, y_prob in probs.items():
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    linhas.append({
        "modelo": nome,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "AUC": roc_auc_score(y_test, y_prob)
    })

pd.DataFrame(linhas).sort_values("AUC", ascending=False)

### Atividade

Com base na tabela anterior:

1. O modelo com maior AUC teve também menos erros relevantes?
2. Qual modelo você escolheria se o principal objetivo fosse reduzir FN?
3. Qual modelo parece mais simples de justificar para uma decisão de crédito?

## 20. Overfitting e validação cruzada

Modelos mais flexíveis podem se ajustar demais aos dados de treino.

Para avaliar melhor a estabilidade, usamos validação cruzada.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_resultados = []

for nome, modelo in modelos.items():
    scores = cross_val_score(
        modelo,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )
    cv_resultados.append({
        "modelo": nome,
        "AUC_media_CV": scores.mean(),
        "AUC_dp_CV": scores.std()
    })

pd.DataFrame(cv_resultados).sort_values("AUC_media_CV", ascending=False)

### Interpretação

A validação cruzada ajuda a verificar se o desempenho é estável.

Um modelo pode apresentar:

- alta AUC média;
- mas grande variação entre folds.

Isso indica menor estabilidade.

## 21. Quando usar cada modelo?

Resumo prático:

- regressão logística: quando explicação e estabilidade são fundamentais;
- árvore: quando queremos regras simples;
- random forest: quando buscamos robustez;
- gradient boosting: quando buscamos maior desempenho preditivo.

> A escolha depende do contexto, dos dados e do objetivo da decisão.

## 22. Mercado, governança e responsabilidade

Na prática:

- modelos podem ser comprados de bureaus;
- modelos internos podem ser desenvolvidos;
- modelos precisam ser monitorados continuamente.

Mesmo quando se usa um modelo externo:

> a instituição continua responsável pela decisão.

## 23. Extensões

Além da modelagem de PD, existem outras frentes importantes:

- LGD: quanto se perde dado o default;
- EAD: quanto está exposto no momento do default;
- crédito corporativo: risco de empresas;
- risco soberano: risco de países.

Neste minicurso, nosso foco foi o primeiro passo: estimar e usar a probabilidade de inadimplência.

## 24. Atividade final

Escolha um modelo entre:

- logística;
- árvore;
- random forest;
- gradient boosting.

Responda:

1. Qual modelo teve melhor AUC?
2. Qual modelo parece mais interpretável?
3. Qual threshold você usaria?
4. Como você justificaria essa decisão para uma instituição financeira?
5. Que risco existe em usar apenas uma métrica para escolher o modelo?

## 25. Síntese final

Neste notebook, vimos que:

- modelos estimam probabilidades;
- métricas avaliam desempenho;
- thresholds transformam probabilidades em decisões;
- custos ajudam a escolher decisões melhores;
- modelos diferentes podem levar a decisões diferentes.

> Modelar é importante. Decidir é essencial.